In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "py_src")
sys.path.insert(0, "../py_src")
sys.path.insert(0, "../../py_src")

from voxeloo.notebooks.imports import *
from _voxeloo_cpp_ext import voxels_to_mesh

In [ ]:
import json
import numpy as np
from dataclasses import dataclass
from PIL import Image

In [ ]:
@dataclass
class TextureSet:
    top: str
    side: str
    bottom: str
        
@dataclass
class Material:
    name: str
    order: str 
    textures: TextureSet
        
@dataclass
class Material:
    name: str
    order: str 
    textures: TextureSet

In [ ]:
class Texture:
    
    def __init__(self, name):
        self.name = name
        
    def get(self):
        path = f"../../../voxeloo-web/public/textures/{self.name}.png"
        return np.array(Image.open(path))
    
class ColoredTexture:
    
    def __init__(self, name, blend=[1.0, 1.0, 1.0]):
        self.texture = Texture(name) 
        self.blend = blend
        
    def get(self):
        rgba = self.texture.get().astype(np.float32) / 255.0
        rgba[:, :, 0:3] *= self.blend
        return (rgba * 255).astype(np.uint8)

In [ ]:
MATERIALS = [
    Material(
        name="stone",
        order="bst",
        textures=TextureSet(
            top=Texture("stone"),
            side=Texture("stone"),
            bottom=Texture("stone"),
        ),
    ),
    Material(
        name="spruce",
        order="bst",
        textures=TextureSet(
            top=Texture("spruce_planks"),
            side=Texture("spruce_planks"),
            bottom=Texture("spruce_planks"),
        ),
    ),
    Material(
        name="oak",
        order="bst",
        textures=TextureSet(
            top=Texture("oak_log_top"),
            side=Texture("oak_log"),
            bottom=Texture("oak_log_top"),
        ),
    ),
    Material(
        name="grass",
        order="bst",
        textures=TextureSet(
            top=Texture("grass_block_top"),
            side=Texture("grass_block_side"),
            bottom=Texture("dirt"),
        ),
    ),
    Material(
        name="spruce_leaves",
        order="bst",
        textures=TextureSet(
            top=ColoredTexture("spruce_leaves", blend=[0.4, 0.8, 0.5]),
            side=ColoredTexture("spruce_leaves", blend=[0.4, 0.8, 0.5]),
            bottom=ColoredTexture("spruce_leaves", blend=[0.4, 0.8, 0.5]),
        ),
    ),
    Material(
        name="oak_leaves",
        order="bst",
        textures=TextureSet(
            top=ColoredTexture("oak_leaves", blend=[0.1, 1.0, 0.2]),
            side=ColoredTexture("oak_leaves", blend=[0.1, 1.0, 0.2]),
            bottom=ColoredTexture("oak_leaves", blend=[0.1, 1.0, 0.2]),
        ),
    ),
]

In [ ]:
def material(name):
    return next(m for m in MATERIALS if m.name == name)

def to_rgba(array):
    r = array[:, :, :, 0].astype(np.uint32) << 24
    g = array[:, :, :, 1].astype(np.uint32) << 16
    b = array[:, :, :, 2].astype(np.uint32) << 8
    a = array[:, :, :, 3].astype(np.uint32)
    return r | g | b | a

### Block painting routines

In [ ]:
def paint_block_face(out, face, pixels):
    if face == "x_neg":
        out[0, :, :, :] = pixels
    elif face == "x_pos":
        out[-1, :, :, :] = pixels[:, ::-1]
    elif face == "y_neg":
        out[:, 0, :, :] = pixels
    elif face == "y_pos":
        out[:, -1, :, :] = pixels
    elif face == "z_neg":
        out[:, :, 0, :] = np.swapaxes(pixels[:, ::-1], 0, 1)
    elif face == "z_pos":
        out[:, :, -1, :] = np.swapaxes(pixels, 0, 1)

def to_block(name):
    m = material(name)
    
    top = m.textures.top.get()
    side = m.textures.side.get()[::-1, :, :]
    bottom = m.textures.bottom.get()
    
    sides = [
        ("x_neg", side, "s"),
        ("x_pos", side, "s"),
        ("y_neg", bottom, "b"),
        ("y_pos", top, "t"),
        ("z_neg", side, "s"),
        ("z_pos", side, "s"),
    ]
    
    sides.sort(key=lambda side: m.order.find(side[2]))
    
    out = np.zeros(shape=(16, 16, 16, 4), dtype=np.uint32)
    for side in sides:
        paint_block_face(out, side[0], side[1])
    return to_rgba(out)

#### Render grass block

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_block("grass"))
)

#### Render stone block

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_block("stone"))
)

#### Render oak block

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_block("oak"))
)

#### Render oak leaves block

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_block("oak_leaves"))
)

In [ ]:
voxels_to_mesh(to_block("oak_leaves")).vertices.shape

#### Render spruce leaves block

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_block("spruce_leaves"))
)

### Projection routines

In [ ]:
def shift_mask(mask, face):
    d, h, w = mask.shape
    if face == "x_neg":
        return np.concatenate([np.zeros((16, 16, 1)), mask[:, :, :-1]], axis=2)
    elif face == "x_pos":
        return np.concatenate([mask[:, :, 1:], np.zeros((16, 16, 1))], axis=2)  
    elif face == "y_neg":
        return np.concatenate([np.zeros((16, 1, 16)), mask[:, :-1, :]], axis=1)
    elif face == "y_pos":
        return np.concatenate([mask[:, 1:, :], np.zeros((16, 1, 16))], axis=1)
    elif face == "z_neg":
        return np.concatenate([np.zeros((1, 16, 16)), mask[:-1, :, :]], axis=0)
    elif face == "z_pos":
        return np.concatenate([mask[1:, :, :], np.zeros((1, 16, 16))], axis=0)
    
def project_pixels(shape, pixels, face):
    d, h, w = shape
    if face in ["x_neg", "x_pos"]:
        return np.repeat(np.swapaxes(pixels[:, :, np.newaxis, :], 0, 1), w, axis=2)
    elif face in ["y_neg", "y_pos"]:
        return np.repeat(pixels[:, np.newaxis, :, :], h, axis=1)
    elif face in ["z_neg", "z_pos"]:
        return np.repeat(pixels[np.newaxis, :, :, :], d, axis=0)

def paint_shape_face(out, face, pixels, mask):
    face_mask = np.logical_and(mask == 1, shift_mask(mask, face) == 0)
    face_proj = project_pixels(mask.shape, pixels, face)
    out[face_mask] = face_proj[face_mask]
    return out

def to_shape(name, mask):
    m = material(name)
    
    top = m.textures.top.get()
    side = m.textures.side.get()[::-1, :, :]
    bottom = m.textures.bottom.get()
    
    sides = [
        ("x_neg", side, "s"),
        ("x_pos", side, "s"),
        ("y_neg", bottom, "b"),
        ("y_pos", top, "t"),
        ("z_neg", side, "s"),
        ("z_pos", side, "s"),
    ]
    
    sides.sort(key=lambda side: m.order.find(side[2]))
    
    out = np.zeros(shape=(16, 16, 16, 4), dtype=np.uint32)
    for side in sides:
        paint_shape_face(out, side[0], side[1], mask)
    return to_rgba(out)

#### Define shapes

In [ ]:
def stairs():
    out = np.zeros(shape=(16, 16, 16), dtype=bool)
    out[:, :, :] = True
    out[8:, 8:, :] = False
    return out

def plank():
    out = np.zeros(shape=(16, 16, 16), dtype=bool)
    out[:, :, :] = True
    out[:, 8:, :] = False
    return out

def fence():
    out = np.zeros(shape=(16, 16, 16), dtype=bool)
    out[:, :, :] = False
    out[6:10, :, 0:4] = True
    out[6:10, :, 12:16] = True
    out[7:9, 12:15, :] = True
    out[7:9, 5:8, :] = True
    return out

def table():
    out = np.zeros(shape=(16, 16, 16), dtype=bool)
    out[:, :, :] = False
    out[:, -2:, :] = True
    out[:2, :, :2] = True
    out[-2:, :, :2] = True
    out[:2, :, -2:] = True
    out[-2:, :, -2:] = True
    return out

In [ ]:
visualize_3d_window(
    voxels_to_mesh(
        np.repeat(to_block("stone")[:, :, 0:1], 16, axis=2)
    )
)

#### Render steps

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("stone", stairs())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("spruce", stairs())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("oak", stairs())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("grass", stairs())),
)

#### Render planks

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("stone", plank())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("spruce", plank())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("oak", plank())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("grass", plank())),
)

#### Render fences

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("stone", fence())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("spruce", fence())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("oak", fence())),
)

In [ ]:
voxels_to_mesh(to_shape("oak", fence()))

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("grass", fence())),
)

#### Render table

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("stone", table())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("spruce", table())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("oak", table())),
)

In [ ]:
visualize_3d_window(
    voxels_to_mesh(to_shape("grass", table())),
)